In [344]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Data Scrapping and Modification

In [65]:
city_urls = {
    "Hyderabad": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Hyderabad",
    "Bangalore": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Bangalore",
    "Mumbai": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Mumbai",
    "Pune": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Pune",
    "Chennai": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Chennai"
}

# Define headers to prevent a browser from blocking
headers = {'User-Agent': 'Mozilla/5.0'}

max_properties_per_city = 300

# list to store all scraped data
all_data = []

# Keywords to detect actual property types from title text
property_keywords = ['Apartment', 'Flat', 'Studio', 'Villa', 'House', 'Floor', 'Penthouse', 'Row House', 'Farm']

# Loop through each city and its corresponding URL
for city, base_url in city_urls.items():
    print(f"\nScraping {city}...")
    city_data = []   # Store city-specific listings
    page = 1      # Start from page 1

    # Loop through pages until desired property count is reached
    while len(city_data) < max_properties_per_city:
        url = f"{base_url}&page={page}"
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')

        # Find all property listing cards on the page
        listings = soup.find_all("div", class_="mb-srp__card")
        if not listings:
            print(" No listings found, stopping.")
            break

        # Loop through each property card
        for listing in listings:
            try:
                # Extract bhk from title
                title_tag = listing.find("h2", class_="mb-srp__card--title")
                title = title_tag.get_text(strip=True)

                bhk_match = re.search(r"(\d+)\s*BHK", title.upper())
                bhk = bhk_match.group(1) if bhk_match else None

                # Extract location
                locality = title.split("in", 1)[-1].strip() if "in" in title else None

                # Detect property type from title
                title_clean = title.replace(",", " ").replace("-", " ")
                title_words = title_clean.split()
                prop_type_from_title = next((word for word in title_words if word.capitalize() in property_keywords), None)
            except:
                bhk = locality = prop_type_from_title = None

            # Extract rent price
            try:
                rent = listing.find("div", class_="mb-srp__card__price--amount").get_text(strip=True)
                rent = rent.replace("₹", "").replace(",", "").strip()
            except:
                rent = None

            summary = listing.find("div", class_="mb-srp__card__summary__list")
            summary_items = summary.find_all("div", class_="mb-srp__card__summary__list--item") if summary else []

            # Initializing Variable
            area = furnishing = facing = property_type = overlooking = Bathroom = Balcony=TenantPreferred=Availability= None
            
            # Loop through summary items and extract specific features
            for item in summary_items:
                try:
                    label = item.find("div", class_="mb-srp__card__summary--label").get_text(strip=True).lower()
                    value = item.find("div", class_="mb-srp__card__summary--value").get_text(strip=True)

                    if "area" in label:
                        area = value
                    elif "furnish" in label:
                        furnishing = value
                    elif "facing" in label:
                        facing = value
                    elif "property type" in label or "type" in label:
                        property_type = value
                    elif "overlooking" in label :
                        overlooking = value
                    elif "bathroom" in label:
                        Bathroom = value  
                    elif "balcony" in label:
                        Balcony = value 
                    elif "tenant preferred" in label:
                        TenantPreferred=value
                    elif "availability" in label:
                        Availability=value
                        

                except:
                    continue

            # Final fallback if property_type is missing or invalid
            if not property_type or property_type.lower() in ["for", "in", "rent", "sale"]:
                property_type = prop_type_from_title

            if not bhk or not locality:
                continue

            city_data.append({
                "City": city,
                "BHK": bhk.strip(),
                "Location": locality.strip(),
                "Price (₹)": rent,
                "Area (sqft)": area.strip() if area else None,
                "Property Type": property_type.strip() if property_type else None,
                "Furnishing": furnishing.strip() if furnishing else None,
                "Property Facing": facing.strip() if facing else None,
                "overlooking":overlooking.strip() if overlooking else None,
                "Bathroom":Bathroom.strip() if Bathroom else None,
                "Balcony":Balcony.strip() if Balcony else None,
                "Tenant Preferred":TenantPreferred.strip() if TenantPreferred else None,
                "Availability":Availability.strip() if Availability else None,

                
            })

            if len(city_data) >= max_properties_per_city:
                break

        page += 1    # Move to next Page
        time.sleep(1)  # delay to avoid overloading the server


    # Append city data to the master list
    all_data.extend(city_data)
print("\nScraping completed...")


Scraping Hyderabad...

Scraping Bangalore...

Scraping Mumbai...

Scraping Pune...

Scraping Chennai...

Scraping completed...


## Data Handling

In [228]:
# Save to CSV
df = pd.DataFrame(all_data)
df.to_csv("MagicBricksProject_Scraped_Data.csv", index=False)
print("\nData saved to MagicBricksProject_Scraped_Data.csv")


Data saved to MagicBricksProject_Scraped_Data.csv


In [229]:
# DataFrame overview
df.head(5)

,City,BHK,Location,Price (₹),Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
0,Hyderabad,3,"Prestige Tranquil, Kokapet, Outer Ring Road, H...",85000,1335 sqft,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,2,None,None
1,Hyderabad,1,"Kondapur, Hyderabad",25000,450 sqft,Flat,Furnished,East,None,1,1,Bachelors/Family,Immediately
2,Hyderabad,3,"SMR Vinay Iconia, Kondapur, Hyderabad",60000,1805 sqft,Flat,Unfurnished,West,"Garden/Park, Pool, Main Road",3,2,Bachelors,Immediately
3,Hyderabad,4,"Kokapet, Outer Ring Road Hyderabad",2.3 Lac,5030 sqft,Villa,Semi-Furnished,East,Garden/Park,5,1,Bachelors/Family,Immediately
4,Hyderabad,3,"Rajapushpa Provincia, Kokapet, Outer Ring Road...",85000,2335 sqft,Flat,Semi-Furnished,East,"Garden/Park, Main Road",3,2,Bachelors/Family,Immediately


In [230]:
# Number of Columns
print("Number of columns:",df.shape[1])

Number of columns: 13


In [231]:
# Number of  Rows

print("Number of rows:",df.shape[0])

Number of rows: 1500


In [232]:
# Print the data types of each column
print("Data types of each column:\n",df.dtypes)

Data types of each column:
 City                object
BHK                 object
Location            object
Price (₹)           object
Area (sqft)         object
Property Type       object
Furnishing          object
Property Facing     object
overlooking         object
Bathroom            object
Balcony             object
Tenant Preferred    object
Availability        object
dtype: object


In [334]:
# Print the number of missing values in each column
print("Missing values in each column:\n",df.isnull().sum())

Missing values in each column:
 City                  0
BHK                   0
Location              0
Price (₹)             0
Area (sqft)           0
Property Type         0
Furnishing            0
Property Facing       0
overlooking           0
Bathroom              0
Balcony             330
Tenant Preferred     10
Availability         10
dtype: int64


In [234]:
# It shows the duplicated rows count
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns): {num_duplicates}")

Total duplicate rows (all columns): 17


In [235]:
#Duplicate data
df[df.duplicated]

,City,BHK,Location,Price (₹),Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
91,Hyderabad,3,"Bandlaguda Jagir, Hyderabad",41000,1320 sqft,Flat,Semi-Furnished,East,"Garden/Park, Pool",2,2,Bachelors,Immediately
96,Hyderabad,4,"MIGH Colony Mehdipatnam, Hyderabad",27000,1600 sqft,Flat,Semi-Furnished,East,Main Road,2,None,Bachelors,Immediately
519,Bangalore,3,L And T Olivia At Raintree Boulevard Cluster 7...,77000,1453 sqft,Flat,Semi-Furnished,East,Garden/Park,3,2,Bachelors/Family,Immediately
533,Bangalore,1,"House Of Hiranandani Bannerghatta, Bannerughat...",34000,400 sqft,Flat,Semi-Furnished,West,Garden/Park,1,1,Bachelors/Family,Immediately
534,Bangalore,1,"Sobha Lake Garden, Seegehalli Krishnarajapura,...",39000,900 sqft,Flat,Furnished,East,Garden/Park,1,1,Bachelors/Family,Immediately
548,Bangalore,2,Pragathi Layout Doddanekundi Bangalore,35000,1200 sqft,House,Semi-Furnished,None,Main Road,2,2,Bachelors/Family,Immediately
549,Bangalore,3,"Prestige Finsbury Park Regent, Bagalur Main Ro...",35000,1562 sqft,Flat,Semi-Furnished,None,None,3,None,Bachelors/Family,Immediately
550,Bangalore,3,"Sai Purvi Symphony, Gunjur, Bangalore",50000,1550 sqft,Flat,Semi-Furnished,West,Main Road,3,1,Bachelors/Family,Immediately
564,Bangalore,4,Sarjapur Road Bangalore,55000,3853 sqft,Villa,Semi-Furnished,North - East,Garden/Park,4,2,Bachelors,Immediately
681,Mumbai,3,"Oberoi Eternia And Enigma, Oberoi Eternia And ...",90000,1049 sqft,Flat,Semi-Furnished,East,Main Road,3,None,Bachelors/Family,Immediately


In [236]:
df.drop_duplicates(inplace=True)

In [237]:
# Drop duplicate
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns): {num_duplicates}")

Total duplicate rows (all columns): 0


In [238]:
print("Number of rows:",df.shape[0])

Number of rows: 1483


## Data Cleaning

In [369]:
df.isna().sum()

City                 0
BHK                  0
Location             0
Price (₹)            0
Area (sqft)          0
Property Type        0
Furnishing           0
Property Facing      0
overlooking          0
Bathroom             0
Balcony              0
Tenant Preferred     0
Availability        10
dtype: int64

In [ ]:
# # List of locations 
# localities = [
#     "Kondapur", "Narsingi", "Gachibowli", "Tellapur", "Kompally", "Kollur", "Shamshabad","Nalagandla",
#     "Bachupally", "Miyapur", "Hitech City", "Puppalaguda", "Kokapet", "Manikonda", "Madhapur",
#     "Whitefield", "Varthur", "Hebbal", "Yelahanka", "Devanahalli",
#     "Sarjapur", "Electronic City", "Kanakapura Road", "Bannerghatta Main Road","Rajajinagar",
#     "Mulund", "Andheri", "Chembur", "Worli", "Powai", "Borivali", "Bandra", "Wadala",
#     "Goregaon", "Kandivali East", "Thakur Village", "Juhu", "Kandivali", "Hadapsar", "Mundhwa",
#     "Kharadi", "Baner", "Hadapsar", "Hinjawadi", "Wagholi", "Wakad", "Balewadi",
#     "NIBM Road", "Viman Nagar", "Undri", "Sholinganallur", "Medavakkam", "Pallikaranai", "Porur", "Perungudi", 
#     "Pallavaram", "Guduvancheri", "Kelambakkam", "Thiruvanmiyur", "Padur"
    
# ]

# # Lowercased for comparison
# localities_lower = [loc.lower() for loc in localities]

# # Function to clean location
# def clean_location(text):
#     if pd.isna(text):
#         return ''
#     text = re.sub(r'\s+', ' ', text).lower()
#     for loc, loc_lower in zip(localities, localities_lower):
#         if loc_lower in text:
#             return loc
#     return np.nan

# # Update Location column in-place
# df['Location'] = df['Location'].apply(clean_location)  

In [240]:
# List of locations 
localities = [
    "Kondapur", "Narsingi", "Gachibowli", "Tellapur", "Kompally", "Kollur", "Shamshabad","Nalagandla",
    "Bachupally", "Miyapur", "Hitech City", "Puppalaguda", "Kokapet", "Manikonda", "Madhapur",
    "Whitefield", "Varthur", "Hebbal", "Yelahanka", "Devanahalli",
    "Sarjapur", "Electronic City", "Kanakapura Road", "Bannerghatta Main Road","Rajajinagar",
    "Mulund", "Andheri", "Chembur", "Worli", "Powai", "Borivali", "Bandra", "Wadala",
    "Goregaon", "Kandivali East", "Thakur Village", "Juhu", "Kandivali", "Hadapsar", "Mundhwa",
    "Kharadi", "Baner", "Hadapsar", "Hinjawadi", "Wagholi", "Wakad", "Balewadi",
    "NIBM Road", "Viman Nagar", "Undri", "Sholinganallur", "Medavakkam", "Pallikaranai", "Porur", "Perungudi", 
    "Pallavaram", "Guduvancheri", "Kelambakkam", "Thiruvanmiyur", "Padur"
    # ---------------- PUNE ----------------
    "Shivaji Nagar", "Deccan Gymkhana", "Camp", "Swargate", "Kothrud", "Aundh",
    "Baner", "Bavdhan", "Pashan", "Warje", "Viman Nagar", "Koregaon Park",
    "Kalyani Nagar", "Kharadi", "Hadapsar", "Wagholi", "Mundhwa", "Bibwewadi",
    "Katraj", "Dhayari", "Hinjewadi", "Wakad", "Pimple Saudagar", "Pimpri",

    # ---------------- HYDERABAD ----------------
    "Charminar", "Malakpet", "Nampally", "Asif Nagar", "Gachibowli",
    "HITEC City", "Madhapur", "Kondapur", "Financial District", "Nanakramguda",
    "Kukatpally", "Miyapur", "Chanda Nagar", "Hafeezpet", "Bachupally",
    "Nizampet", "Manikonda", "Narsingi", "Uppal", "Begumpet",
    "Banjara Hills", "Jubilee Hills", "Punjagutta", "Secunderabad",

    # ---------------- MUMBAI ----------------
    "Andheri", "Andheri East", "Andheri West", "Bandra", "Bandra East", "Bandra West",
    "Borivali", "Borivali East", "Borivali West", "Dadar", "Dadar East", "Dadar West",
    "Goregaon", "Goregaon East", "Goregaon West", "Juhu", "Kandivali",
    "Kandivali East", "Kandivali West", "Malad", "Malad East", "Malad West",
    "Lower Parel", "Worli", "Powai", "Chembur", "Ghatkopar", "Kurla",
    "Thane", "Navi Mumbai", "Vashi", "Nerul", "Kharghar",

    # ---------------- CHENNAI ----------------
    "T Nagar", "Adyar", "Velachery", "Anna Nagar", "Tambaram", "Chromepet",
    "Guindy", "Porur", "Vadapalani", "Nungambakkam", "Mylapore", "Kodambakkam",
    "Royapettah", "Egmore", "Triplicane", "Thiruvanmiyur", "Perungudi",
    "Sholinganallur", "OMR", "ECR", "Pallavaram", "Medavakkam",

    # ---------------- BANGALORE ----------------
    "Whitefield", "Marathahalli", "KR Puram", "Indiranagar", "Koramangala",
    "HSR Layout", "BTM Layout", "Electronic City", "Bellandur", "Sarjapur Road",
    "MG Road", "Jayanagar", "JP Nagar", "Banashankari", "Yelahanka",
    "Hebbal", "Malleshwaram", "Rajajinagar", "Basavanagudi", "Kengeri",
    "Nagarbhavi", "Vijayanagar"
]

# Lowercased for comparison
localities_lower = [loc.lower() for loc in localities]

# Function to clean location
def clean_location(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'\s+', ' ', text).lower()
    for loc, loc_lower in zip(localities, localities_lower):
        if loc_lower in text:
            return loc
    return np.nan

# Update Location column in-place
df['Location'] = df['Location'].apply(clean_location)  

In [241]:
#Fixing Nan value with Empty
df["Location"]=df["Location"].fillna(" ")

In [242]:
print(df["Furnishing"].unique())
df[df["Furnishing"].isna()]

['Semi-Furnished' 'Furnished' 'Unfurnished' None]


,City,BHK,Location,Price (₹),Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
127,Hyderabad,4,,40000,2418 sqft,Villa,None,None,None,3,2,Bachelors/Family,Immediately
1381,Chennai,2,Guduvancheri,9000,875 sqft,Flat,None,North,Main Road,None,None,Bachelors,Immediately


In [ ]:
df.groupby(by=["Furnishing"])["Furnishing"].agg(["count"])

,count
Furnishing,
Furnished,335
Semi-Furnished,854
Unfurnished,292


In [253]:
df["Furnishing"].mode()[0]

'Semi-Furnished'

In above Group-by table we can see that, Semi-Furnished	has more count then Furnished and Unfurnished. I have decieded to assign not available values with maximum repeating value,i.e is "Semi-Furnished".

In [245]:
#fixed with "Semi-Furnished"
df["Furnishing"]=df["Furnishing"].fillna("Semi-Furnished")

In [260]:
df.groupby("Property Facing")["Property Facing"].value_counts()

Property Facing
East            757
North           171
North - East    103
North - West     24
South            40
South - East      9
South -West       9
West            151
Name: count, dtype: int64

In [267]:
df["Property Facing"].mode()[0]

'East'

In [264]:
df["Property Facing"]=df["Property Facing"].fillna(df["Property Facing"].mode()[0])

In [270]:
df["overlooking"].unique()

array(['Pool, Garden/Park, Main Road', None,
       'Garden/Park, Pool, Main Road', 'Garden/Park',
       'Garden/Park, Main Road', 'Main Road', 'Pool', 'Garden/Park, Pool',
       'Pool, Main Road', 'Main Road, Garden/Park, Pool',
       'Main Road, Garden/Park', 'Pool, Garden/Park',
       'Pool, Main Road, Garden/Park', 'Garden/Park, Main Road, Pool'],
      dtype=object)

In [273]:
df["overlooking"].isna().sum()

np.int64(316)

In [ ]:
df.groupby("overlooking")["overlooking"].value_counts()


overlooking
Garden/Park                     299
Garden/Park, Main Road          189
Garden/Park, Main Road, Pool      1
Garden/Park, Pool                56
Garden/Park, Pool, Main Road    152
Main Road                       340
Main Road, Garden/Park           13
Main Road, Garden/Park, Pool     14
Pool                             18
Pool, Garden/Park                18
Pool, Garden/Park, Main Road     58
Pool, Main Road                   8
Pool, Main Road, Garden/Park      1
Name: count, dtype: int64

In [276]:
print(df["overlooking"].mode()[0])
df["overlooking"]=df["overlooking"].fillna(df["overlooking"].mode()[0])

Main Road


In [333]:
df["Bathroom"]=df["Bathroom"].fillna(df["Bathroom"].mode()[0])

In [337]:
df.groupby("Balcony")["Balcony"].value_counts()

Balcony
1     462
10      1
2     478
3     164
4      43
5       4
6       1
Name: count, dtype: int64

In [340]:
df["Balcony"]=df["Balcony"].fillna("Unknown")

In [359]:

df.groupby("Tenant Preferred")["Tenant Preferred"].value_counts()

Tenant Preferred
Bachelors           482
Bachelors/Family    726
Family              265
Name: count, dtype: int64

In [366]:
df["Tenant Preferred"]=df["Tenant Preferred"].fillna(df["Tenant Preferred"].mode()[0])

In [370]:
df["Availability"]=df["Availability"].fillna(df["Availability"].mode()[0])

In [371]:
df.isna().sum()

City                0
BHK                 0
Location            0
Price (₹)           0
Area (sqft)         0
Property Type       0
Furnishing          0
Property Facing     0
overlooking         0
Bathroom            0
Balcony             0
Tenant Preferred    0
Availability        0
dtype: int64

In [387]:
df.describe()

,BHK,Bathroom
count,1483.000000,1483.000000
mean,2.637222,2.628456
std,0.892624,0.980333
min,1.000000,1.000000
25%,2.000000,2.000000
50%,3.000000,3.000000
75%,3.000000,3.000000
max,6.000000,7.000000


In [382]:
df=pd.read_csv("Cleaned_Data.csv")

In [383]:
df.drop(columns="Unnamed: 0",inplace=True)


In [386]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1483 entries, 0 to 1482
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   City              1483 non-null   object
 1   BHK               1483 non-null   int64 
 2   Location          1483 non-null   object
 3   Price (₹)         1483 non-null   object
 4   Area (sqft)       1483 non-null   object
 5   Property Type     1483 non-null   object
 6   Furnishing        1483 non-null   object
 7   Property Facing   1483 non-null   object
 8   overlooking       1483 non-null   object
 9   Bathroom          1483 non-null   int64 
 10  Balcony           1483 non-null   object
 11  Tenant Preferred  1483 non-null   object
 12  Availability      1483 non-null   object
dtypes: int64(2), object(11)
memory usage: 150.7+ KB


In [388]:
df.head(1)

,City,BHK,Location,Price (₹),Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
0,Hyderabad,3,Kokapet,85000,1335 sqft,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,2,Bachelors/Family,Immediately


## Outliers